<a href="https://colab.research.google.com/github/sc22lg/my_pizza/blob/EG_version/CRT_eval_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## aCFG model test

### Setup

In [47]:
# Import stuff
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
import tqdm

import random
import time

from pathlib import Path
import pickle
import os

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"
import plotly.graph_objects as go

from torch.utils.data import DataLoader

from functools import *
import pandas as pd
import gc

# import comet_ml
import itertools
# import comet_ml
import wandb
import itertools
class HookPoint(nn.Module):
    def __init__(self):
        super().__init__()
        self.fwd_hooks = []
        self.bwd_hooks = []
    def give_name(self, name):
        self.name = name
    def add_hook(self, hook, dir='fwd'):
        def full_hook(module, module_input, module_output):
            return hook(module_output, name=self.name)
        if dir=='fwd':
            handle = self.register_forward_hook(full_hook)
            self.fwd_hooks.append(handle)
        elif dir=='bwd':
            handle = self.register_backward_hook(full_hook)
            self.bwd_hooks.append(handle)
        else:
            raise ValueError(f"Invalid direction {dir}")
    def remove_hooks(self, dir='fwd'):
        if (dir=='fwd') or (dir=='both'):
            for hook in self.fwd_hooks:
                hook.remove()
            self.fwd_hooks = []
        if (dir=='bwd') or (dir=='both'):
            for hook in self.bwd_hooks:
                hook.remove()
            self.bwd_hooks = []
        if dir not in ['fwd', 'bwd', 'both']:
            raise ValueError(f"Invalid direction {dir}")
    def forward(self, x):
        return x

class Embed(nn.Module):
    def __init__(self, d_vocab, d_model):
        super().__init__()
        self.W_E = nn.Parameter(torch.randn(d_model, d_vocab)/np.sqrt(d_model))
    def forward(self, x):
        return torch.einsum('dbp -> bpd', self.W_E[:, x])

class Unembed(nn.Module):
    def __init__(self, d_vocab, d_model):
        super().__init__()
        self.W_U = nn.Parameter(torch.randn(d_model, d_vocab)/np.sqrt(d_vocab))
    def forward(self, x):
        return (x @ self.W_U)

# Positional Embeddings
class PosEmbed(nn.Module):
    def __init__(self, max_ctx, d_model):
        super().__init__()
        self.W_pos = nn.Parameter(torch.randn(max_ctx, d_model)/np.sqrt(d_model))
    def forward(self, x):
        return x+self.W_pos[:x.shape[-2]]

# Attention
class Attention(nn.Module):
    def __init__(self, d_model, num_heads, d_head, n_ctx, attn_coeff):
        super().__init__()
        self.W_K = nn.Parameter(torch.randn(num_heads, d_head, d_model)/np.sqrt(d_model))
        self.W_Q = nn.Parameter(torch.randn(num_heads, d_head, d_model)/np.sqrt(d_model))
        self.W_V = nn.Parameter(torch.randn(num_heads, d_head, d_model)/np.sqrt(d_model))
        self.W_O = nn.Parameter(torch.randn(d_model, d_head * num_heads)/np.sqrt(d_model))
        self.attn_coeff = attn_coeff
        self.register_buffer('mask', torch.tril(torch.ones((n_ctx, n_ctx))))
        self.d_head = d_head
        self.hook_k = HookPoint()
        self.hook_q = HookPoint()
        self.hook_v = HookPoint()
        self.hook_z = HookPoint()
        self.hook_attn = HookPoint()
        self.hook_attn_pre = HookPoint()

    def forward(self, x):
        k = self.hook_k(torch.einsum('ihd,bpd->biph', self.W_K, x))
        q = self.hook_q(torch.einsum('ihd,bpd->biph', self.W_Q, x))
        v = self.hook_v(torch.einsum('ihd,bpd->biph', self.W_V, x))
        attn_scores_pre = torch.einsum('biph,biqh->biqp', k, q)
        attn_scores_masked =attn_scores_pre
        attn_matrix = self.hook_attn(
            F.softmax(self.hook_attn_pre(attn_scores_masked/np.sqrt(self.d_head)), dim=-1)\
            *self.attn_coeff+(1-self.attn_coeff))
        z = self.hook_z(torch.einsum('biph,biqp->biqh', v, attn_matrix))
        z_flat = einops.rearrange(z, 'b i q h -> b q (i h)')
        out = torch.einsum('df,bqf->bqd', self.W_O, z_flat)
        return out

class MLP(nn.Module):
    def __init__(self, d_model, d_mlp, act_type):
        super().__init__()
        self.W_in = nn.Parameter(torch.randn(d_mlp, d_model)/np.sqrt(d_mlp))
        self.b_in = nn.Parameter(torch.zeros(d_mlp))
        self.W_out = nn.Parameter(torch.randn(d_model, d_mlp)/np.sqrt(d_model))
        self.b_out = nn.Parameter(torch.zeros(d_model))
        self.act_type = act_type
        # self.ln = LayerNorm(d_mlp, model=self.model)
        self.hook_pre = HookPoint()
        self.hook_post = HookPoint()
        assert act_type in ['ReLU', 'GeLU', 'Tanh']

    def forward(self, x):
        x = self.hook_pre(torch.einsum('md,bpd->bpm', self.W_in, x) + self.b_in)
        if self.act_type=='ReLU':
            x = F.relu(x)
        elif self.act_type=='GeLU':
            x = F.gelu(x)
        elif self.act_type=='Tanh':
            x = F.tanh(x)
        x = self.hook_post(x)
#        return x
        x = torch.einsum('dm,bpm->bpd', self.W_out, x) + self.b_out
        return x

# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_head, num_heads, n_ctx, act_type, attn_coeff):
        super().__init__()
        self.attn = Attention(d_model, num_heads, d_head, n_ctx, attn_coeff=attn_coeff)
        self.mlp = MLP(d_model, d_model*4,act_type)
        self.hook_attn_out = HookPoint()
        self.hook_mlp_out = HookPoint()
        self.hook_resid_pre = HookPoint()
        self.hook_resid_mid = HookPoint()
        self.hook_resid_post = HookPoint()

    def forward(self, x):
        x = self.hook_resid_mid(x + self.hook_attn_out(self.attn(self.hook_resid_pre(x))))
        x = self.hook_resid_post(x + self.hook_mlp_out(self.mlp(x)))
        return x

# Full transformer
class Transformer(nn.Module):
    def __init__(self, num_layers, d_vocab, d_model, d_head, num_heads, n_ctx, act_type, attn_coeff, use_cache=False, use_ln=True):
        super().__init__()
        assert 0<=attn_coeff<=1
        #print('parameters', num_layers, d_vocab, d_model, d_head, num_heads, n_ctx, act_type, attn_coeff, use_cache, use_ln)
        self.cache = {}
        self.use_cache = use_cache

        self.embed = Embed(d_vocab, d_model)
        self.pos_embed = PosEmbed(n_ctx, d_model)
        self.unembed = Unembed(d_vocab, d_model)
        self.use_ln = use_ln
        self.blocks = nn.ModuleList([TransformerBlock(d_model, d_head, num_heads, n_ctx, act_type, attn_coeff) for i in range(num_layers)])

        for name, module in self.named_modules():
            if type(module)==HookPoint:
                module.give_name(name)

    def forward(self, x):
        x = self.embed(x)
        x = self.pos_embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.unembed(x)
        return x
    def forward_h(self, x):
        x = self.embed(x)
        tmp=x
        x = self.pos_embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.unembed(x)
        return tmp,x
    def forward_b(self, x):
        x = self.embed(x)
        tmp=x
        x = self.pos_embed(x)
        for blk in self.blocks:
            x = blk(x)
        return x

    def set_use_cache(self, use_cache):
        self.use_cache = use_cache

    def hook_points(self):
        return [module for name, module in self.named_modules() if 'hook' in name]

    def remove_all_hooks(self):
        for hp in self.hook_points():
            hp.remove_hooks('fwd')
            hp.remove_hooks('bwd')

    def cache_all(self, cache, incl_bwd=False):
        # Caches all activations wrapped in a HookPoint
        def save_hook(tensor, name):
            cache[name] = tensor.detach()
        def save_hook_back(tensor, name):
            cache[name+'_grad'] = tensor[0].detach()
        for hp in self.hook_points():
            hp.add_hook(save_hook, 'fwd')
            if incl_bwd:
                hp.add_hook(save_hook_back, 'bwd')

    def parameters_norm(self):
        # Returns the l2 norm of all parameters
        return sum([torch.sum(p*p).item() for p in self.parameters()])**0.5

    def l2_norm(self):
        # Returns the l2 norm of all parameters
        return sum([torch.sum(p*p) for p in self.parameters()])

    def parameters_flattened(self):
        # Returns all parameters as a single tensor
        return torch.cat([p.view(-1) for p in self.parameters()]).detach().cpu().numpy()
#DEVICE='cuda'
DEVICE='cpu'
print(f'Running on {DEVICE}')
class MyAddDataSet(torch.utils.data.Dataset):
    def __init__(self, func, C, diff_vocab=False, eqn_sign=False):
        self.func = func
        dim = 2
        self.dim = dim
        self.C = C
        self.inputs = []
        self.outputs = []
        self.vocab=C
        if diff_vocab:
            self.vocab*=2
        if eqn_sign:
            self.vocab+=1
            self.dim+=1
        self.vocab_out=0
        for p in range(C**dim):
            x = np.unravel_index(p, (C,)*dim)
            o=self.func(x)
            s=[x[0],x[1]]
            if diff_vocab:
                s[1]+=C
            if eqn_sign:
                s.append(self.vocab-1)
            self.inputs.append(s)
            self.outputs.append(o)
            self.vocab_out=max(self.vocab_out, o+1)
        if self.vocab_out!=C:
            print(f'warning {self.vocab_out=} neq to {C=}')
        self.inputs = torch.tensor(self.inputs, dtype=torch.long, device=DEVICE)
        self.outputs = torch.tensor(self.outputs, dtype=torch.long, device=DEVICE)
        #print(self.inputs,self.outputs)
    def __len__(self):
        return len(self.outputs)
    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

def cross_entropy_high_precision(logits, labels):
    # Shapes: batch x vocab, batch
    # Cast logits to float64 because log_softmax has a float32 underflow on overly
    # confident data and can only return multiples of 1.2e-7 (the smallest float x
    # such that 1+x is different from 1 in float32). This leads to loss spikes
    # and dodgy gradients
    logprobs = F.log_softmax(logits.to(torch.float64), dim=-1)
    prediction_logprobs = torch.gather(logprobs, index=labels[:, None], dim=-1)
    loss = -torch.mean(prediction_logprobs)
    return loss
def run_experiment(config, silent=False):
    exp_name=config['name']
    if not silent:
        print('parsing func',config['funcs'])
    config['func']=eval(config['funcs'])
    full_dataset = MyAddDataSet(func=config['func'],C=config['C'],diff_vocab=config['diff_vocab'],eqn_sign=config['eqn_sign'])
    model = Transformer(
        num_layers=config.get('n_layers',1),
        num_heads=config['n_heads'],
        d_model=config['d_model'],
        d_head=config.get('d_head',config['d_model']//config['n_heads']),
        attn_coeff=config['attn_coeff'],
        d_vocab=full_dataset.vocab,
#        attention_dir=config.get('attention_dir','bidirectional'),
        act_type=config.get('act_fn','relu'),
        n_ctx=full_dataset.dim,
#        normalization_type=None,
    )
    model.to(DEVICE)
    train_size = int(config['frac'] * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])
    if not silent:
        print('random split',len(train_dataset),len(test_dataset))
    batch_size = config.get('batch_size',len(full_dataset))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    opt = optim.AdamW(model.parameters(),lr=config.get('lr',1e-3),weight_decay=config.get('weight_decay',1e-4),betas=(0.9,0.98))
    scheduler = optim.lr_scheduler.LambdaLR(opt, lambda step: min(step/10, 1)) # 10 epoch warmup
    if not silent:
        print(config.get('lr',1e-3),config.get('weight_decay',1e-4))
        print(opt,scheduler)
    losses=[]
    accs=[]
    losses_val=[]
    accs_val=[]
    norms=[]
    loss_val=10
    acc_val=0
    stop=None
    best_train_acc=0.
    best_test_acc=0.
    perfect_train_time=None
    perfect_test_time=None
    pbar = range(config.get('epoch',10000))
    if not silent:
        pbar=tqdm.tqdm(pbar)
    gaps=[]
    early_stop_a=2
    early_stop_b=1
    if config.get('early_stop',None) is not None:
        early_stop_a, early_stop_b = config['early_stop']
    early_stop_timer=0
    #model.train()
    run = None#wandb.init(reinit=True,config=config,project='modadd_new')#,settings=wandb.Settings(start_method="spawn"))
    try:
        for i in pbar:
            def evaluation():
                nonlocal best_test_acc
                nonlocal perfect_test_time
                nonlocal early_stop_timer
                nonlocal early_stop_a
                nonlocal early_stop_b
                # evaluate on test set, return loss and accuracy
                # with torch.inference_mode():
                    #model.eval()
                losses_eval=[]
                accs_eval=[]
                for inp,ans in test_loader:
                    # print(inp.shape)
                    out = model(inp)[:,-1,:]
                    loss = cross_entropy_high_precision(out,ans)
                    acc = torch.sum((out.argmax(dim=1)==ans).float())/len(ans)
                    # print(inp,'test',out.argmax(dim=1),ans)
#                    acc = (out.argmax(dim=1)==ans).float().mean()
                    losses_eval.append(loss.item())
                    accs_eval.append(acc.item())
                    # print(loss,acc)
                #print(losses_eval,accs_eval)
                eval_loss, eval_acc = np.mean(losses_eval), np.mean(accs_eval)
                best_test_acc = max(best_test_acc, eval_acc)
                if eval_acc==1. and perfect_test_time is None:
                    perfect_test_time = i
                if eval_acc>=early_stop_a:
                    early_stop_timer+=1
                else:
                    early_stop_timer=0
                #print(eval_loss,eval_acc)
                return eval_loss, eval_acc
            if early_stop_timer>=early_stop_b:
                break
            for inp,ans in train_loader:
                #print(inp.shape,inp.dtype)
                # print(inp,'train')
                #print(len(inp))
                #model.train()
                out = model(inp)[:,-1,:]
                loss = cross_entropy_high_precision(out,ans)
                loss_val, acc_val = evaluation()
                #print(loss_val,acc_val)
                loss.backward()
                # clip gradients
                #if config.get('clip',None) is not None:
                #    nn.utils.clip_grad_norm_(model.parameters(), config['clip'])
                opt.step()
                scheduler.step()
                opt.zero_grad()
                acc = (out.argmax(dim=1)==ans).float().mean()
                norm = sum([torch.sum(p*p).item() for p in model.parameters()])**0.5
                #sum(p.norm()**2 for p in model.parameters()).sqrt().item()
                losses.append(loss.item())
                accs.append(acc.item())
                losses_val.append(loss_val)
                accs_val.append(acc_val)
                norms.append(norm)

                best_train_acc=max(best_train_acc,acc.item())
                if acc.item()==1. and perfect_train_time is None:
                    perfect_train_time = i
                gaps.append(best_train_acc-best_test_acc)
                if pbar is tqdm.tqdm:
                    pbar.set_description(f"loss: {loss.item():.3f}, accm: {best_train_acc:.3f}, vloss: {loss_val:.3f}, vaccm: {best_test_acc:.3f}, norm: {norm:.3f}, acc: {acc.item():.3f}, vacc: {acc_val:.3f}")
                #print(f"loss: {loss.item():.3f}, accm: {best_train_acc:.3f}, vloss: {loss_val:.3f}, vaccm: {best_test_acc:.3f}, norm: {norm:.3f}, acc: {acc.item():.3f}, vacc: {acc_val:.3f}")
                if run:
                    run.log({'training_loss': loss.item(),
                    'validation_loss': loss_val,
                    'training_accuracy': acc.item(),
                    'validation_accuracy': acc_val,
                    'parameter_norm': norm,
                    'best_train_accuracy': best_train_acc,
                    'best_test_accuracy': best_test_acc,
                    'generalization_gap': best_train_acc-best_test_acc,
                    'generalization_delay1': sum(gaps)})
    except KeyboardInterrupt:
        print('Keyboard interrupt. Gracefully exiting...')
        pass
    if not silent:
        print('Finished.')
    generalization_gap=best_train_acc-best_test_acc
    generalization_delay1=sum(gaps)
    generalization_delay2=sum(max(t-(best_train_acc-best_test_acc),0) for t in gaps)
    if run:
        run.summary["generalization_delay2"] = generalization_delay2
    # run.finish()
    return dict(
        losses=losses,
        accs=accs,
        losses_val=losses_val,
        accs_val=accs_val,
        norms=norms,
        model=model,
        config=config,
        generalization_gap=generalization_gap,
        generalization_delay1=generalization_delay1,
        generalization_delay2=generalization_delay2,
        best_train_acc=best_train_acc,
        best_test_acc=best_test_acc,
        perfect_train_time=perfect_train_time,
        perfect_test_time=perfect_test_time,
        dataset = full_dataset,
        run=run
    )

Running on cpu


### aCRT test

In [48]:
from scipy.optimize import curve_fit
import pandas as pd
import numpy as np # Ensure numpy is imported

def run_r_squared_experiment(model, dataset):
    C = dataset.C
    n_neurons = 512 # Assuming a fixed number of neurons

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=C * C)

    # Cache activations
    ch = {}
    for x, y in dataloader:
        with torch.inference_mode():
            model.remove_all_hooks()
            model.cache_all(ch)
            model.eval()
            model(x)
            model.remove_all_hooks()
        break # Only need one batch to get activations

    cached_mlp_pre = ch['blocks.0.mlp.hook_pre']
    mlp_pre = cached_mlp_pre[:, 1, :]
    mlp_pre = mlp_pre.reshape(C, C, n_neurons)

    def sinusoid(a, amplitude, freq, phase, offset):
        return amplitude * np.cos(2 * np.pi * freq * a / C + phase) + offset

    a_values = np.arange(C, dtype=float)
    candidate_freqs = np.arange(1, C // 2 + 1)
    fixed_b = 0

    results = []

    for neuron_idx in range(n_neurons):
        y = mlp_pre[:, fixed_b, neuron_idx].numpy()

        best_r2 = -np.inf
        best_freq = None
        best_params = None

        for f in candidate_freqs:
            try:
                p0 = [y.std(), f, 0.0, y.mean()]
                params, _ = curve_fit(
                    lambda a, amp, phase, offset: sinusoid(a, amp, f, phase, offset),
                    a_values, y,
                    p0=[p0[0], p0[2], p0[3]],
                    maxfev=5000
                )
                y_pred = sinusoid(a_values, params[0], f, params[1], params[2])
                ss_res = np.sum((y - y_pred) ** 2)
                ss_tot = np.sum((y - y.mean()) ** 2)
                r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

                if r2 > best_r2:
                    best_r2 = r2
                    best_freq = f
                    best_params = params
            except RuntimeError:
                continue

        results.append({
            'neuron': neuron_idx,
            'best_freq': best_freq,
            'best_r2': best_r2,
            'params': best_params,
        })

    r2_values = np.array([r['best_r2'] for r in results])

    # Calculate the number of unique frequencies 'learnt' (e.g., R2 > 0.85)
    learnt_freqs = set(r['best_freq'] for r in results if r['best_r2'] > 0.85 and r['best_freq'] is not None)
    num_learnt_freqs = len(learnt_freqs)

    return r2_values, num_learnt_freqs



# Dynamically read model IDs
model_type = "GD"
num_models = 100

model_ids = [f'{model_type}_{i}' for i in range(1, num_models + 1)]

all_model_r2_data = []

for runid in model_ids:
    print(f"Processing model: {runid}")
    config_file = f'save/config_{runid}.json'
    model_path = f'save/model_{runid}.pt'

    # Check if config and model files exist before proceeding
    if not os.path.exists(config_file):
        print(f"Config file not found for {runid}: {config_file}. Skipping.")
        continue
    if not os.path.exists(model_path):
        print(f"Model file not found for {runid}: {model_path}. Skipping.")
        continue

    with open(config_file, 'r') as f:
        config = json.load(f)

    config.setdefault('diff_vocab', False)
    config.setdefault('eqn_sign', False)

    # Initialize dataset based on config (similar to how run_experiment does it)
    dataset = MyAddDataSet(func=eval(config['funcs']), C=config['C'],
                             diff_vocab=config['diff_vocab'], eqn_sign=config['eqn_sign'])

    # Initialize model architecture
    model = Transformer(
        num_layers=config.get('n_layers', 1),
        num_heads=config['n_heads'],
        d_model=config['d_model'],
        d_head=config.get('d_head', config['d_model'] // config['n_heads']),
        attn_coeff=config['attn_coeff'],
        d_vocab=dataset.vocab,
        act_type=config.get('act_fn', 'relu'),
        n_ctx=dataset.dim,
    )
    model.to(DEVICE)

    # Load trained weights
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))

    r2_values, num_learnt_freqs = run_r_squared_experiment(model, dataset)

    all_model_r2_data.append({
        'model_id': runid,
        'mean_r2': r2_values.mean(),
        'median_r2': np.median(r2_values),
        'frac_r2_gt_0.85': (r2_values > 0.85).mean(),
        'frac_r2_gt_0.95': (r2_values > 0.95).mean(),
        'num_learnt_freqs': num_learnt_freqs
    })

r2_df = pd.DataFrame(all_model_r2_data)
r2_df.to_csv(f'{model_type}_model_r_squared_summary.csv', index=False)

print("R-squared summary saved to model_r_squared_summary.csv")
print(r2_df)

Processing model: GD_1


/tmp/ipykernel_2955/828713634.py:45: OptimizeWarning:

Covariance of the parameters could not be estimated



Processing model: GD_2
Processing model: GD_3
Processing model: GD_4
Processing model: GD_5
Processing model: GD_6
Processing model: GD_7
Processing model: GD_8
Processing model: GD_9
Processing model: GD_10
Processing model: GD_11
Processing model: GD_12
Processing model: GD_13
Processing model: GD_14
Processing model: GD_15
Processing model: GD_16
Processing model: GD_17
Processing model: GD_18
Processing model: GD_19
Processing model: GD_20
Processing model: GD_21
Processing model: GD_22
Processing model: GD_23
Processing model: GD_24
Processing model: GD_25
Processing model: GD_26
Processing model: GD_27
Processing model: GD_28
Processing model: GD_29
Processing model: GD_30
Processing model: GD_31
Processing model: GD_32
Processing model: GD_33
Processing model: GD_34
Processing model: GD_35
Processing model: GD_36
Processing model: GD_37
Processing model: GD_38
Processing model: GD_39
Processing model: GD_40
Processing model: GD_41
Processing model: GD_42
Processing model: GD_43


In [50]:
import pandas as pd

model_type = "GD"
summary_df = pd.read_csv(f'{model_type}_model_r_squared_summary.csv')

print("Summary DataFrame:")
print(summary_df)
print("\nMeans of each category:")
print(summary_df.mean(numeric_only=True))

Summary DataFrame:
   model_id   mean_r2  median_r2  frac_r2_gt_0.85  frac_r2_gt_0.95  \
0      GD_1  0.908747   0.976630         0.806641         0.771484   
1      GD_2  0.829152   0.955702         0.638672         0.523438   
2      GD_3  0.966564   0.981104         0.964844         0.906250   
3      GD_4  0.905349   0.969621         0.810547         0.662109   
4      GD_5  0.923823   0.971284         0.851562         0.707031   
..      ...       ...        ...              ...              ...   
95    GD_96  0.951360   0.979564         0.925781         0.892578   
96    GD_97  0.902123   0.975192         0.810547         0.695312   
97    GD_98  0.969999   0.985358         0.960938         0.927734   
98    GD_99  0.928613   0.980700         0.875000         0.794922   
99   GD_100  0.963442   0.982685         0.955078         0.919922   

    num_learnt_freqs  
0                  3  
1                  4  
2                  4  
3                  5  
4                  4  
..